<a href="https://colab.research.google.com/github/accoelhos/Trabalho_Teoria_Computacao/blob/main/trabalho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Definição do Problema

## 1.1 Informações Importantes nos Estados

Os estados do autômato precisam rastrear


*   Contexto base: se ele está lendo algo decimal (0-9), haxadecimal (A-F) ou científico.
*   Estágio da parte numérica: se o dígito que ele leu pertence à parte principal do número, ao expoente (em caso de científico) ou à parte imaginária (em caso de número complexo).
* Flags de símbolos: deve ter conhecimento de prefixos como 0x e #, sufixos como h, i e separadores como ., e, E.

## 1.2 Representação como linguagem L

A lingaugem do trabalho é a união de cinco sublinguagens

$$L = L_{int} \cup L_{real} \cup L_{sci} \cup L_{hex} \cup L_{comp}$$

Com o alfabeto $\Sigma = \{0..9, A..F, a..f, ., +, -, e, E, x, h, i, \#\}$.

## 1.3 Estratégia para solução do problema

Implementar um AFD Único com um conjunto de AFDs em paralelo.

* Se começar com 0x ou #, o fluxo é desviado para a lógica Hexadecimal

* Se encontrar um . ou e, o fluxo evolui de inteiro para real ou científico

* Se no final de uma expressão numérica surgir um + ou - seguido de novos dígitos e um i, o fluxo valida um número complexo.

# 2. Definição Formal do AF

## 2.1 Descrição dos estados ($Q$)

| Estado | Propósito | Categoria de Aceitação |
| :--- | :--- | :--- |
| **q0** | **Estado Inicial**: Ponto de partida da leitura. | - |
| **q1** | **Sinal**: Após leitura de sinal inicial (+ ou -). | - |
| **q2** | **Inteiro**: Lendo dígitos decimais. | **Inteiros** |
| **q3** | **Ponto Isolado**: Após leitura de "." inicial (ex: .5). | - |
| **q4** | **Real**: Lendo dígitos da parte fracionária. | **Reais** |
| **q5** | **Expoente**: Após o caractere de notação científica (e, E). | - |
| **q6** | **Sinal do Expoente**: Após + ou - dentro da parte científica. | - |
| **q7** | **Científico**: Lendo dígitos do expoente. | **Notação Científica** |
| **q8** | **Prefixo Hexa**: Após detectar "0x", "0X" ou "#". | - |
| **q9** | **Hexadecimal**: Lendo dígitos de base 16 (0-9, A-F). | **Hexadecimal** |
| **q10** | **Complexo**: Após validar o sufixo "i" da parte imaginária. | **Complexos** |
| **qerr** | **Erro**: Estado de rejeição para sequências inválidas. | - |

## 2.2 Alfabeto de entrada ($\Sigma$)

$$\Sigma = \{0..9, A..F, a..f, ., +, -, e, E, x, h, i, \#\}$$

### 2.3 Descrição da Função de Transição ($\delta$)
Abaixo, a tabela de transição que mapeia o comportamento do autômato.
*Nota: Consideramos **d** como dígitos {0-9} e **h** como dígitos hexadecimais {0-9, A-F, a-f}.*

| Estado Atual | Símbolo ($\sigma$) | Próximo Estado | Descrição da Transição |
| :--- | :--- | :--- | :--- |
| **q0** | {+, -} | **q1** | Permite sinal opcional antes dos dígitos. |
| **q0** | d | **q2** | Início da cadeia por dígito. |
| **q0** | . | **q3** | Início por ponto decimal (ex: .123). |
| **q0** | {#} | **q8** | Prefixo hexadecimal via tralha. |
| **q1** | d | **q2** | Dígitos após sinal. |
| **q2** | d | **q2** | Loop de leitura de parte inteira. |
| **q2** | . | **q4** | Transição para ponto flutuante (ex: 5.). |
| **q2** | {e, E} | **q5** | Transição para notação científica. |
| **q3** | d | **q4** | Algarismos após o ponto inicial. |
| **q4** | d | **q4** | Loop de leitura de parte fracionária. |
| **q4** | {e, E} | **q5** | Real evoluindo para científico. |
| **q5** | {+, -} | **q6** | Sinal opcional do expoente. |
| **q5** | d | **q7** | Primeiro dígito do expoente. |
| **q7** | d | **q7** | Loop do expoente. |
| **q8** | h | **q9** | Início de dígitos hexa após prefixo. |
| **q9** | h | **q9** | Loop de dígitos hexadecimais. |
| **q2, q4, q7** | {+, -} | **q1** | Início da parte imaginária (reusa lógica de sinal/dígito). |
| **q2, q4, q7** | {i} | **q10** | Identifica número imaginário puro ou fim de complexo. |

### 2.4 Estados Finais ($F$)
O conjunto de estados de aceitação é definido por:
$$F = \{q_2, q_4, q_7, q_9, q_{10}\}$$

# 3. Implementação

- Código completo da classe ```FiniteAutomata```
- Funções auxiliares



In [18]:
from typing import Dict, Tuple, Optional
num = ["1", "2", "3", "4", "5", "6", "7", "8", "9"]
sinal = ["-", "+"]
ponto = ['.']
img = ['i']
euler = ['E', 'e']
letra = ['A', 'B', 'C', 'D', 'F', 'a', 'b', 'c', 'd', 'f']
prefixo1 = ['0']
prefixo2 = ['x']
prefixo3 = ['#']
zero = ['0']
sufixo = ['h']

class FiniteAutomata:
  delta: Dict[Tuple[str, str], str] = None

  def __init__(self, tape):
    # Inicialização
    self.tape = tape
    self.head = 0
    self.estado_atual = 'q0'

    self.delta = {
        ('q0', 'n'): 'q1',
        ('q0', 's'): 'q2',
        ('q0', '0'): 'q3',
        ('q0', '.'): 'q4',

        ('q1', 'n'): 'q1',
        ('q1', '0'): 'q1',
        ('q1', '.'): 'q6',
        ('q1', 'e'): 'q13',

        ('q2', 'n'): 'q1',
        ('q2', '0'): 'q1',

        ('q3', 'n'): 'q1',
        ('q3', '0'): 'q1',
        ('q3', 'x'): 'q16',

        ('q4', 'n'): 'q5',
        ('q4', '0'): 'q5',

        ('q5', 'n'): 'q5',
        ('q5', '0'): 'q5',
        ('q5', 'e'): 'q13',

        ('q6', 'n'): 'q7',
        ('q6', '0'): 'q7',
        ('q6', 'e'): 'q13',

        ('q7', 'n'): 'q7',
        ('q7', '0'): 'q7',
        ('q7', 'e'): 'q13',

        ('q7', 's'): 'q8',

        ('q8', 'n'): 'q9',
        ('q8', '0'): 'q9',


        ('q9', 'n'): 'q9',
        ('q9', '0'): 'q9',
        ('q9', '.'): 'q10',
        ('q9', 'i'): 'q12',

        ('q10', 'n'): 'q11',
        ('q10', '0'): 'q11',

        ('q11', 'n'): 'q11',
        ('q11', '0'): 'q11',
        ('q11', 'i'): 'q12',

        ('q13', 's'): 'q14',
        ('q13', 'n'): 'q15',
        ('q13', '0'): 'q15',

        ('q14', 'n'): 'q15',
        ('q14', '0'): 'q15',

        ('q15', 'n'): 'q15',
        ('q15', '0'): 'q15',

        ('q16', 'n'): 'q17',
        ('q16', '0'): 'q17',
        ('q16', 'l'): 'q17',
        ('q16', 'e'): 'q17',

        ('q17', '0'): 'q17',
        ('q17', 'n'): 'q17',
        ('q17', 'l'): 'q17',
        ('q17', 'e'): 'q17',
        ('q17', 'h'): 'q18'
    }

    self.estado_final = {'q1', 'q3','q6', 'q5', 'q7', 'q17', 'q18', 'q15', 'q12'}
    self.estado_erro   = 'qerr'

  def step(self):
      """
      Executa um passo do AF
      Retorna: keep_running
      """

      try:
        self.tape = str(self.tape)
      except E:
        print('A sequencia não pode ser convertida em string')
        self.estado_atual = 'qerr'
        return False

      simbolo = self.tape[self.head]
      if simbolo in num:
        simbolo = 'n'
      if simbolo in letra:
        simbolo = 'l'
      if simbolo in euler:
        simbolo = 'e'
      if simbolo in sinal:
        simbolo = 's'
      transicao = (self.estado_atual, simbolo)
      self.estado_atual = self.delta.get(transicao, self.estado_erro)
      self.head += 1

      keep_running = (self.head < len(self.tape) and self.estado_atual != self.estado_erro)
      return keep_running

  def execute(self):
      """
      Executa o AF para a fita fornecida
      Retorna: (accept)
      """
      if len(self.tape) <= 0:
        return False

      while self.step():
        pass

      accept = self.estado_atual in self.estado_final

      return accept

# 4. Testes

- Casos pequenos
- Verificação de correção


In [19]:
casos = [
    ("123",       True),
    ("-456",      True),
    ("+789",      True),
    ("0",         True),
    ("3.14",      True),
    ("-0.5",      True),
    ("+2.0",      True),
    (".5",        True),
    ("5.",        True),
    ("1.23e4",    True),
    ("-5E-10",    True),
    ("6e+7",      True),
    (".5e3",      True),
    ("2.e-2",     True),
    ("0xFF",      True),
    ("0x1A3",     True),
    ("3+4i",      True),
    ("2.5-1.3i",  True),

    ("",          False),   # fita vazia
    ("+",         False),   # só sinal, sem dígito
    ("--5",       False),   # dois sinais


]

for fita, esperado in casos:
    af      = FiniteAutomata(fita)
    result  = af.execute()
    status  = "Acertou" if result == esperado else "Errou"
    print(f"{status}  '{fita!s:<8}' → {'aceite' if result else 'rejeita'}  (esperado: {'aceite' if esperado else 'rejeita'})")

Acertou  '123     ' → aceite  (esperado: aceite)
Acertou  '-456    ' → aceite  (esperado: aceite)
Acertou  '+789    ' → aceite  (esperado: aceite)
Acertou  '0       ' → aceite  (esperado: aceite)
Acertou  '3.14    ' → aceite  (esperado: aceite)
Acertou  '-0.5    ' → aceite  (esperado: aceite)
Acertou  '+2.0    ' → aceite  (esperado: aceite)
Acertou  '.5      ' → aceite  (esperado: aceite)
Acertou  '5.      ' → aceite  (esperado: aceite)
Acertou  '1.23e4  ' → aceite  (esperado: aceite)
Acertou  '-5E-10  ' → aceite  (esperado: aceite)
Acertou  '6e+7    ' → aceite  (esperado: aceite)
Acertou  '.5e3    ' → aceite  (esperado: aceite)
Acertou  '2.e-2   ' → aceite  (esperado: aceite)
Acertou  '0xFF    ' → aceite  (esperado: aceite)
Acertou  '0x1A3   ' → aceite  (esperado: aceite)
Errou  '3+4i    ' → rejeita  (esperado: aceite)
Acertou  '2.5-1.3i' → aceite  (esperado: aceite)
Acertou  '        ' → rejeita  (esperado: rejeita)
Acertou  '+       ' → rejeita  (esperado: rejeita)
Acertou  '--5    

# 5. Demonstração Formal

- Prova de corretude do AF apresentado

# 6. Conclusão

- Principais aprendizados
- Limitações do estudo